# Figure S16

Compares AEM resistivity among the three groundwater response classes.


In [ ]:
from pathlib import Path
import warnings

import json
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy as raster_xy

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023.')


def display_path(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
LABELS_PATH = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
AEM_PATH = ROOT / 'data' / '1 resistivity' / 'aem_log10res_1km_masked.tif'
AEM_DEPTH_PATH = ROOT / 'data' / '1 resistivity' / 'aem_depth_levels_m.json'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS16'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESISTIVITY_FIG_PATH = OUT_DIR / 'FigS16_ab_resistivity.png'

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9.0,
    'axes.labelsize': 9.5,
    'axes.titlesize': 10.0,
    'xtick.labelsize': 8.3,
    'ytick.labelsize': 8.3,
    'legend.fontsize': 8.2,
    'axes.linewidth': 0.75,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
})

In [ ]:
labels = pd.read_csv(LABELS_PATH)
required = {
    'grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class',
}
missing = required.difference(labels.columns)
if missing:
    raise KeyError(f'Missing required label columns: {sorted(missing)}')

labels['valid_for_clustering'] = labels['valid_for_clustering'].astype(bool)
valid = labels['valid_for_clustering'].to_numpy() & labels['response_class'].notna().to_numpy()
rows = labels['row'].to_numpy(dtype=np.int64)
cols = labels['col'].to_numpy(dtype=np.int64)
response_class = labels['response_class'].to_numpy()

depth_m = np.asarray(json.loads(AEM_DEPTH_PATH.read_text(encoding='utf-8')), dtype=np.float32)

with rasterio.open(AEM_PATH) as source:
    if source.count != len(depth_m):
        raise ValueError(f'AEM bands ({source.count}) != depth levels ({len(depth_m)})')
    raster_rows = source.height - 1 - rows
    inside = (
        (raster_rows >= 0) & (raster_rows < source.height)
        & (cols >= 0) & (cols < source.width)
    )
    if not np.all(inside):
        raise ValueError('Some class-label cells fall outside the AEM raster.')
    x_check, y_check = raster_xy(source.transform, raster_rows, cols, offset='center')
    coordinate_error_m = np.nanmax(np.abs(
        np.column_stack([x_check, y_check])
        - labels[['x', 'y']].to_numpy(dtype=float)
    ))
    if coordinate_error_m > 1e-3:
        raise ValueError(f'AEM coordinate mismatch: {coordinate_error_m:.3f} m')
    aem_stack = source.read().astype(np.float32)
    if source.nodata is not None:
        aem_stack[aem_stack == source.nodata] = np.nan
    log10_profile = aem_stack[:, raster_rows, cols].T

rho_profile = np.power(10.0, log10_profile).astype(np.float32)


def depth_mask(zmin: float, zmax: float) -> np.ndarray:
    return (depth_m >= zmin) & (depth_m < zmax)


DEPTH_BANDS_M = [
    (0, 15), (15, 30), (30, 50), (50, 75), (75, 100),
    (100, 150), (150, 200), (200, 250), (250, 300), (300, 350),
]
RHO_BIN_EDGES = np.asarray([0, 5, 10, 20, 30, np.inf], dtype=float)
RHO_BIN_LABELS = ['0–5', '5–10', '10–20', '20–30', '≥30']
RHO_BIN_COLORS = ['#2C6DB2', '#78B7D4', '#F1E6A7', '#E99A57', '#B94A3D']

resistivity_composition = {}
resistivity_coverage = {}
for class_name in CLASS_ORDER:
    class_cells = valid & (response_class == class_name)
    class_values = rho_profile[class_cells]
    composition_rows = []
    coverage_rows = []
    for depth_min, depth_max in DEPTH_BANDS_M:
        band_values = class_values[:, depth_mask(depth_min, depth_max)]
        finite_values = band_values[np.isfinite(band_values)]
        counts, _ = np.histogram(finite_values, bins=RHO_BIN_EDGES)
        composition_rows.append(
            100.0 * counts / counts.sum() if counts.sum()
            else np.full(5, np.nan)
        )
        coverage_rows.append(float(np.isfinite(band_values).mean()))
    composition_array = np.asarray(composition_rows, dtype=float)
    if not np.allclose(np.nansum(composition_array, axis=1), 100.0, atol=1e-6):
        raise ValueError(f'Resistivity fractions do not close to 100% for {class_name}.')
    resistivity_composition[class_name] = composition_array
    resistivity_coverage[class_name] = np.asarray(coverage_rows, dtype=float)

print(f'Coordinate check max error: {coordinate_error_m:.6f} m')
print(f'Valid classified cells: {valid.sum():,}')
print(f'AEM profile: {log10_profile.shape[1]} layers, {depth_m.min():.1f}–{depth_m.max():.1f} m')

In [ ]:
def class_mask(class_name: str) -> np.ndarray:
    return valid & (response_class == class_name)


def style_axis(ax: plt.Axes, grid_axis: str | None = None) -> None:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
        spine.set_color('#333333')
    ax.tick_params(length=3.0, width=0.7, direction='out')
    if grid_axis is not None:
        ax.grid(axis=grid_axis, color='#E5E5E5', linewidth=0.45, zorder=0)
        ax.set_axisbelow(True)


fig_resistivity = plt.figure(figsize=(10.2, 5.1), facecolor='white')
resistivity_grid = fig_resistivity.add_gridspec(
    1, 2, width_ratios=[1.08, 2.42], wspace=0.32,
    left=0.075, right=0.985, bottom=0.185, top=0.94,
)
ax_a = fig_resistivity.add_subplot(resistivity_grid[0, 0])
b_grid = resistivity_grid[0, 1].subgridspec(1, 3, wspace=0.12)
ax_b = [fig_resistivity.add_subplot(b_grid[0, idx]) for idx in range(3)]

for class_name in CLASS_ORDER:
    values = rho_profile[class_mask(class_name)]
    median = np.nanmedian(values, axis=0)
    color = CLASS_COLORS[class_name]
    ax_a.plot(median, depth_m, color=color, linewidth=2.0, label=class_name, solid_capstyle='round')
ax_a.set_xscale('log')
ax_a.set_xlim(2.5, 50)
ax_a.set_ylim(350, 0)
ax_a.set_xticks([2.5, 5, 10, 20, 50])
ax_a.set_xticklabels(['2.5', '5', '10', '20', '50'])
ax_a.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax_a.set_yticks(np.arange(0, 351, 50))
ax_a.set_xlabel('Resistivity (Ω·m)')
ax_a.set_ylabel('Depth (m)')
ax_a.legend(loc='lower right', frameon=False, handlelength=1.8)
style_axis(ax_a, 'both')

depth_centres = np.asarray([(zmin + zmax) / 2 for zmin, zmax in DEPTH_BANDS_M], dtype=float)
depth_heights = np.asarray([0.86 * (zmax - zmin) for zmin, zmax in DEPTH_BANDS_M], dtype=float)
depth_labels = [f'{zmin}–{zmax}' for zmin, zmax in DEPTH_BANDS_M]
for class_idx, (axis, class_name) in enumerate(zip(ax_b, CLASS_ORDER)):
    composition = resistivity_composition[class_name]
    left = np.zeros(len(DEPTH_BANDS_M), dtype=float)
    for bin_idx, (bin_label, bin_color) in enumerate(zip(RHO_BIN_LABELS, RHO_BIN_COLORS)):
        axis.barh(
            depth_centres, composition[:, bin_idx], left=left, height=depth_heights,
            color=bin_color, edgecolor='white', linewidth=0.45, label=bin_label,
        )
        left += composition[:, bin_idx]
    axis.set_xlim(0, 100)
    axis.set_ylim(350, 0)
    axis.set_xticks([0, 50, 100])
    axis.set_title(class_name, color=CLASS_COLORS[class_name], fontsize=8.8, fontweight='bold', pad=4)
    axis.grid(axis='x', color='#E5E5E5', linewidth=0.45, zorder=0)
    axis.set_axisbelow(True)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
        spine.set_color('#333333')
    axis.tick_params(length=2.5, width=0.65, direction='out', labelsize=7.4)
    if class_idx == 0:
        axis.set_yticks(depth_centres)
        axis.set_yticklabels(depth_labels, fontsize=7.2)
        axis.set_ylabel('Depth (m)', labelpad=4)
    else:
        axis.set_yticks(depth_centres)
        axis.set_yticklabels([])
        axis.tick_params(axis='y', length=0)
ax_b[1].set_xlabel('Resistivity fraction (%)', labelpad=3)
legend_handles = [mpl.patches.Patch(facecolor=color, edgecolor='none', label=label)
                  for label, color in zip(RHO_BIN_LABELS, RHO_BIN_COLORS)]
fig_resistivity.legend(
    handles=legend_handles, title='Resistivity (Ω·m)', ncol=5, frameon=False,
    loc='lower center', bbox_to_anchor=(0.695, 0.025), columnspacing=0.8,
    handlelength=1.0, handletextpad=0.35, fontsize=7.1, title_fontsize=7.5,
)

fig_resistivity.savefig(RESISTIVITY_FIG_PATH, dpi=EXPORT_DPI, facecolor='white')
plt.show()

print('Saved:', display_path(RESISTIVITY_FIG_PATH))
print('AEM finite-data coverage by depth interval:')
for class_name in CLASS_ORDER:
    coverage_text = ', '.join(
        f'{depth_min}–{depth_max} m: {100 * coverage:.1f}%'
        for (depth_min, depth_max), coverage in zip(
            DEPTH_BANDS_M, resistivity_coverage[class_name]
        )
    )
    print(f'  {class_name}: {coverage_text}')